<a href="https://colab.research.google.com/github/SohailVibeCoder/IB9AU---GenAI/blob/main/Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Name:** Sohail Essajee (5757504)

Part 1: Sentiment Analysis The sentiment analysis revealed that normal messages have a slightly higher average sentiment polarity (0.146) compared to spam messages (0.113), indicating spam messages tend to be less positive or more neutral on average.

Part 2: Topic Modelling For topic modeling, 5 topics were determined, covering areas such as celebrity news, entertainment, politics, and general conversational content. The distribution of these topics across normal and spam messages was calculated to understand their relationship, though the specific breakdown of this relationship isn't fully detailed in the current output.

----

**Task 1**



----

In [ ]:
# =========================
# PART 1: SENTIMENT ANALYSIS
# =========================

import pandas as pd
import numpy as np
from textblob import TextBlob

# Load data
df = pd.read_csv("/content/data/fakenews.csv")

# Keep only relevant columns and drop missing values
df = df[["text", "label"]].dropna()

# Ensure label is numeric and valid
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df[df["label"].isin([0, 1])]

# Basic text cleaning
df["text_clean"] = (
    df["text"]
    .astype(str)
    .str.lower()
    .str.replace(r"http\S+", "", regex=True)
    .str.replace(r"[^a-z\s]", "", regex=True)
)

# Sentiment function
def sentiment_polarity(text):
    return TextBlob(text).sentiment.polarity

# Compute sentiment
df["sentiment"] = df["text_clean"].apply(sentiment_polarity)

# Aggregate results
sentiment_summary = (
    df.groupby("label")["sentiment"]
    .agg(["mean", "median", "std", "count"])
    .rename(index={0: "Normal", 1: "Spam"})
)

print("=== Sentiment Summary ===")
print(sentiment_summary.round(3))


/tmp/ipython-input-945867582.py:10: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,2

=== Sentiment Summary ===
         mean  median    std  count
label                              
Normal  0.146   0.139  0.124   2925
Spam    0.113   0.108  0.114   1971


In [ ]:
# =========================
# PART 2: TOPIC MODELLING
# =========================

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np

# Vectorization
vectorizer = CountVectorizer(
    stop_words="english",
    max_df=0.9,
    min_df=10,
    max_features=2000
)

X = vectorizer.fit_transform(df["text_clean"])

# Try different topic numbers
topic_range = range(2, 9)
perplexities = []

for k in topic_range:
    lda = LatentDirichletAllocation(
        n_components=k,
        random_state=42,
        learning_method="batch"
    )
    lda.fit(X)
    perplexities.append(lda.perplexity(X))

perplexity_df = pd.DataFrame({
    "n_topics": topic_range,
    "perplexity": perplexities
})

print("=== Perplexity by Number of Topics ===")
print(perplexity_df)


=== Perplexity by Number of Topics ===
   n_topics   perplexity
0         2  1236.700836
1         3  1199.217244
2         4  1137.770375
3         5  1083.597732
4         6  1053.828595
5         7  1029.614813
6         8  1014.070190


In [ ]:
# Final LDA model
n_topics = 5
lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42
)

topic_distributions = lda.fit_transform(X)

# Assign dominant topic
df["dominant_topic"] = topic_distributions.argmax(axis=1)


In [ ]:
feature_names = vectorizer.get_feature_names_out()

print("\n=== Top Words per Topic ===")
for topic_idx, topic in enumerate(lda.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-10:][::-1]]
    print(f"Topic {topic_idx}: {', '.join(top_words)}")



=== Top Words per Topic ===
Topic 0: kardashian, kim, said, new, jenner, brad, kylie, pitt, family, york
Topic 1: film, series, best, new, season, edit, awards, music, year, role
Topic 2: trump, prince, harry, wedding, royal, meghan, markle, president, images, family
Topic 3: said, like, just, people, know, time, im, dont, think, going
Topic 4: love, source, time, just, instagram, couple, relationship, justin, told, new


In [ ]:
# =========================
# Optimised Topic Distribution Table
# =========================

# Compute raw counts
topic_label_dist = (
    df.groupby(["label", "dominant_topic"])
    .size()
    .unstack(fill_value=0)
)

# Convert to percentages
topic_label_pct = (
    topic_label_dist
    .div(topic_label_dist.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

# Rename index for clarity
topic_label_pct.index = ["Normal", "Spam"]

# Rename topic columns
topic_label_pct.columns = [f"Topic {i}" for i in topic_label_pct.columns]

# Add row totals (should sum to ~100)
topic_label_pct["Total (%)"] = topic_label_pct.sum(axis=1)

print("\n=== Topic Distribution by Label (%) ===")
display(topic_label_pct)



=== Topic Distribution by Label (%) ===


,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4,Total (%)
Normal,10.70,19.56,7.73,35.38,26.63,100.0
Spam,15.68,9.18,9.84,28.21,37.09,100.0
